# 🔹 Activity 02: Feature Engineering (Basic)

**Track:** Feature Engineering  
**Level:** Intermediate  
**Prerequisites:** 01_data_preprocessing_pipeline.ipynb

---

## 📖 What You Will Learn
- Encoding strategies (Label, OHE, Ordinal)
- Binning / discretisation
- Creating interaction features
- Extracting features from datetime columns

---

## 🧠 Concept: What Is Feature Engineering?

Feature engineering is the process of using **domain knowledge** to transform raw data into
features that better represent the underlying problem to machine learning algorithms.

> A model can only learn from what you give it. Great features = great models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Ready')

## 📊 Dataset: E-commerce Order Data

In [ ]:
# ── Synthetic e-commerce dataset ─────────────────────────────────────────────
n = 1000
np.random.seed(42)

df = pd.DataFrame({
    'order_date':   pd.date_range('2022-01-01', periods=n, freq='6H'),
    'age':          np.random.randint(18, 65, n),
    'gender':       np.random.choice(['Male', 'Female'], n),
    'category':     np.random.choice(['Electronics','Clothing','Food','Books'], n),
    'membership':   np.random.choice(['Bronze','Silver','Gold','Platinum'], n),
    'quantity':     np.random.randint(1, 10, n),
    'unit_price':   np.random.uniform(5, 500, n).round(2),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20], n),
})

# Target: total revenue
df['revenue'] = (df['quantity'] * df['unit_price'] * (1 - df['discount_pct']/100)
                 + np.random.normal(0, 20, n)).round(2)

print(df.shape)
df.head()

## 🏷️ Section 1 — Encoding Strategies

### 🧠 Concept: When to use which encoding?

| Column | Type | Strategy | Reason |
|---|---|---|---|
| gender | Nominal, binary | Label Encoding | Just 0/1 |
| category | Nominal, 4 levels | One-Hot Encoding | No ordering |
| membership | Ordinal | Ordinal Encoding | Bronze < Silver < Gold < Platinum |

In [ ]:
# ── Label Encoding (binary) ───────────────────────────────────────────────────
le = LabelEncoder()
df['gender_enc'] = le.fit_transform(df['gender'])
print('Gender mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# ── One-Hot Encoding (nominal) ────────────────────────────────────────────────
# WHY: We use get_dummies for quick exploration; in production use sklearn OHE inside a Pipeline.
df_ohe = pd.get_dummies(df['category'], prefix='cat', drop_first=True)
print('\nOHE columns:', df_ohe.columns.tolist())
df = pd.concat([df, df_ohe], axis=1)

# ── Ordinal Encoding (membership tier) ───────────────────────────────────────
membership_order = [['Bronze','Silver','Gold','Platinum']]
oe = OrdinalEncoder(categories=membership_order)
df['membership_enc'] = oe.fit_transform(df[['membership']])
print('\nMembership mapping:')
print(df[['membership','membership_enc']].drop_duplicates().sort_values('membership_enc'))

## 📦 Section 2 — Binning / Discretisation

### 🧠 Concept: Why Bin?

- Captures **non-linear** relationships in tree models
- Reduces impact of outliers
- Creates interpretable segments (e.g., age groups)

> **⚠️ COMMON ERROR:** Binning loses information. Use it purposefully — only when the relationship is truly step-like.

In [ ]:
# ── Equal-width binning ───────────────────────────────────────────────────────
df['age_group'] = pd.cut(
    df['age'],
    bins=[18, 25, 35, 45, 55, 65],
    labels=['18-25','26-35','36-45','46-55','56-65']
)

# ── Equal-frequency (quantile) binning ───────────────────────────────────────
# WHY: Equal-frequency ensures each bin has the same count, preventing class imbalance in buckets.
df['price_tier'] = pd.qcut(
    df['unit_price'],
    q=4,
    labels=['Budget','Economy','Premium','Luxury']
)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['age_group'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Age Group Distribution')
axes[0].set_xlabel('Age Group')

df['price_tier'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Price Tier Distribution (Equal Frequency)')
axes[1].set_xlabel('Tier')

plt.tight_layout()
plt.show()
print(df[['unit_price','price_tier']].head())

## ✖️ Section 3 — Interaction Features

### 🧠 Concept: Interaction Terms

Linear models assume features are **independent**. When the effect of feature A depends on feature B,
we must manually create an **interaction term**: A × B.

Example: `revenue` is driven by `quantity × unit_price`, not by each alone.
A linear model with separate features cannot capture this — but `quantity * unit_price` as a single feature can.

In [ ]:
# ── Manual interaction features ───────────────────────────────────────────────
df['qty_x_price']        = df['quantity'] * df['unit_price']
df['discount_impact']    = df['discount_pct'] / 100 * df['qty_x_price']
df['net_revenue_proxy']  = df['qty_x_price'] - df['discount_impact']
df['price_per_unit_disc']= df['unit_price'] * (1 - df['discount_pct']/100)

# Correlation with target
interaction_cols = ['quantity','unit_price','qty_x_price',
                    'discount_impact','net_revenue_proxy','revenue']
corr = df[interaction_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation: Original vs Interaction Features')
plt.tight_layout()
plt.show()

print('\nCorrelation with revenue:')
print(corr['revenue'].drop('revenue').sort_values(ascending=False))

## 📅 Section 4 — Datetime Feature Extraction

### 🧠 Concept: Time as Features

A datetime column contains many hidden features:
- **Year, Month, Day** → seasonal patterns
- **Weekday** → weekday vs weekend behaviour
- **Hour** → time-of-day effects
- **Is month start/end, quarter** → business cycle effects

> **⚠️ COMMON ERROR:** Passing a datetime object directly to sklearn will raise a TypeError. Always extract numeric features.

In [ ]:
# ── Extract datetime features ─────────────────────────────────────────────────
df['year']         = df['order_date'].dt.year
df['month']        = df['order_date'].dt.month
df['day_of_week']  = df['order_date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['hour']         = df['order_date'].dt.hour
df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter']      = df['order_date'].dt.quarter

# ── Cyclical encoding for periodic features ───────────────────────────────────
# WHY: Month 12 is adjacent to Month 1, but numerically they are far apart.
#      Sin/Cos encoding creates a circular representation.
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)

print('New datetime features:')
dt_cols = ['order_date','year','month','day_of_week','hour','is_weekend',
           'quarter','month_sin','month_cos']
print(df[dt_cols].head())

In [ ]:
# ── Revenue by hour of day ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('hour')['revenue'].mean().plot(ax=axes[0], color='steelblue', marker='o')
axes[0].set_title('Average Revenue by Hour of Day')
axes[0].set_xlabel('Hour')

df.groupby('day_of_week')['revenue'].mean().plot(
    kind='bar', ax=axes[1], color='tomato'
)
axes[1].set_title('Average Revenue by Day of Week')
axes[1].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'], rotation=0)

plt.tight_layout()
plt.show()

## 📈 Section 5 — Compare Models: Before vs After Feature Engineering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── Features BEFORE engineering ───────────────────────────────────────────────
before_cols = ['quantity', 'unit_price', 'discount_pct', 'gender_enc', 'membership_enc']

# ── Features AFTER engineering ────────────────────────────────────────────────
after_cols = before_cols + [
    'qty_x_price', 'net_revenue_proxy', 'discount_impact',
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos',
    'is_weekend', 'quarter',
    'cat_Clothing', 'cat_Electronics', 'cat_Food',
]

# Some OHE cols may not exist in small sample — filter safely
after_cols = [c for c in after_cols if c in df.columns]

y = df['revenue']
results = {}

for label, cols in [('Before FE', before_cols), ('After FE', after_cols)]:
    X_ = df[cols].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(X_, y, test_size=0.2, random_state=42)
    pipe = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
    pipe.fit(X_tr, y_tr)
    r2 = r2_score(y_te, pipe.predict(X_te))
    results[label] = r2
    print(f'{label:12s}  R² = {r2:.4f}')

improvement = (results['After FE'] - results['Before FE']) / abs(results['Before FE']) * 100
print(f'\n🚀 R² improvement: {improvement:+.1f} %')

## 🎯 Summary

| Technique | Purpose | Key Watch-out |
|---|---|---|
| Label Encoding | Binary nominal → 0/1 | Don't use for multi-class nominal |
| One-Hot Encoding | Nominal with k levels | Drop one to avoid dummy trap |
| Ordinal Encoding | Ordered categories | Must specify correct order |
| Binning | Discretise continuous | Information loss; use deliberately |
| Interaction features | Capture combined effects | Can explode feature space |
| Datetime extraction | Reveal temporal patterns | Encode cyclical features as sin/cos |

---

**Next:** [03_feature_engineering_advanced.ipynb](03_feature_engineering_advanced.ipynb)